In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln, digamma
from collections import Counter

# Text Example

In [9]:
import re

with open("alice.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Make period a separate token
text = text.replace(".", " . ")

# Collapse multiple spaces
text = re.sub(r"\s+", " ", text)

tokens = text.strip().split()

print(tokens[:50])

['alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister', 'on', 'the', 'bank', 'and', 'of', 'having', 'nothing', 'to', 'do', '.', 'once', 'or', 'twice', 'she', 'had', 'peeped', 'into', 'the', 'book', 'her', 'sister', 'was', 'reading', 'but', 'it', 'had', 'no', 'pictures', 'or', 'conversations', 'in', 'it', '.', 'and', 'what', 'is', 'the', 'use']


In [10]:
vocab = sorted(set(tokens))
w2i = {w: idx for idx, w in enumerate(vocab)}
i2w = {idx: w for w, idx in w2i.items()}
k = len(vocab)
n = len(tokens)
print(f"Tokens n = {n},  Vocabulary size k = {k}\n")
print("Vocabulary:", vocab)
print("tokens:", tokens)

Tokens n = 2271,  Vocabulary size k = 609

Vocabulary: ['.', 'a', 'about', 'across', 'actually', 'advice', 'advise', 'afraid', 'after', 'afterwards', 'again', 'against', 'air', 'alas', 'alice', 'alices', 'all', 'almost', 'along', 'aloud', 'altogether', 'among', 'an', 'and', 'another', 'answer', 'antipathies', 'anxiously', 'any', 'anything', 'are', 'as', 'ask', 'asking', 'at', 'ate', 'australia', 'away', 'back', 'bank', 'bat', 'bats', 'be', 'beasts', 'beautifully', 'because', 'beds', 'been', 'before', 'began', 'begin', 'beginning', 'begun', 'behind', 'belong', 'best', 'bit', 'bleeds', 'blown', 'book', 'bottle', 'box', 'brave', 'bright', 'brightened', 'bring', 'burn', 'burning', 'burnt', 'but', 'buttered', 'by', 'cake', 'came', 'can', 'candle', 'care', 'cat', 'catch', 'cats', 'center', 'certain', 'certainly', 'chain', 'cheated', 'cherry', 'child', 'children', 'climb', 'close', 'come', 'coming', 'common', 'considering', 'conversation', 'conversations', 'cool', 'corner', 'could', 'couldnt'

## i.i.d Model

The observed counts for each of the $k$ words are given below. 

In [11]:
counts = Counter(tokens)
print("Word counts:")
for w in vocab:
    print(f"  {w:>8s}: {counts[w]}")

Word counts:
         .: 104
         a: 52
     about: 8
    across: 2
  actually: 1
    advice: 1
    advise: 1
    afraid: 1
     after: 5
  afterwards: 1
     again: 4
   against: 1
       air: 2
      alas: 2
     alice: 27
    alices: 1
       all: 10
    almost: 1
     along: 1
     aloud: 1
  altogether: 1
     among: 2
        an: 2
       and: 65
   another: 3
    answer: 1
  antipathies: 1
  anxiously: 1
       any: 4
  anything: 2
       are: 1
        as: 14
       ask: 2
    asking: 1
        at: 11
       ate: 1
  australia: 1
      away: 1
      back: 2
      bank: 1
       bat: 2
      bats: 4
        be: 14
    beasts: 1
  beautifully: 2
   because: 1
      beds: 1
      been: 1
    before: 5
     began: 3
     begin: 1
  beginning: 1
     begun: 2
    behind: 2
    belong: 1
      best: 1
       bit: 2
    bleeds: 1
     blown: 1
      book: 4
    bottle: 4
       box: 2
     brave: 1
    bright: 1
  brightened: 1
     bring: 1
      burn: 1
   burning: 1
     burnt:

We can use the Dirichlet-Multinomial inference here with prior Dirichlet $(0, \dots, 0)$. This leads to the posterior Dirichlet $(x_1, \dots, x_k)$ where $x_i$ is the observed count for the $i$-th word. We can use this fitted model to do sentence generation. We first draw $(p_1, \dots, p_k)$ from Dirichlet $(x_1, \dots, x_k)$ and then draw words from sequentially from Multinomial $(1; p_1, \dots, p_k)$ until we get a '.' (period). 

In [13]:
np.random.seed(42)
# Dirichlet posterior parameters
alpha = np.array([counts[w] for w in vocab], dtype=float)
print("Generating sentences from the i.i.d model posterior:\n")
M = 20 #this is the number of sentences
for s in range(M):
    # Step 1: Draw a probability vector from the Dirichlet posterior
    p = np.random.dirichlet(alpha)    
    # Step 2: Generate words until '.'
    sentence = []
    while True:
        word_idx = np.random.choice(k, p=p)
        word = i2w[word_idx]
        sentence.append(word)
        if word == '.':
            break
    print(f"  Sentence {s+1}: {' '.join(sentence)}")

Generating sentences from the i.i.d model posterior:

  Sentence 1: .
  Sentence 2: in she .
  Sentence 3: .
  Sentence 4: oh herself those this little her herself there growing conversations alas legs little to dark i there those i know for was and was she no .
  Sentence 5: the as row as alice very conversation this of alice what either to but to .
  Sentence 6: generally that was watch bottle to stairs down a empty very poor small drink size .
  Sentence 7: disagree she think to if alice it dinahll fell the again think in on alice girl alice what them it dark was got be nothing was .
  Sentence 8: that fortunately for time a took down on was she ill she had fact you that reading happen .
  Sentence 9: a had ask so not asking .
  Sentence 10: upon and without to she watch side but it after hurt stupid however going one to a how be she and schoolroom fear care was .
  Sentence 11: i middle was tried at but here for rabbit very when seen time neck country neck of ten dear hoping histor

These sentences are of course not realistic, which means that our model is too silly and unrealistic for this dataset. Note that this model works directly with the individual word counts but ignores any information about how words occur in specific sequences. A slighly improved model can be obtained by working with bigram counts as opposed to individual word counts. 

## AR(1) model (also known as the Markov or Bigram model)

This model uses bigram counts (and not the raw data directly). Below we compute the bigram counts $x_{j \mid i}$ as well as $x_i = \sum_{j=1}^k x_{j \mid i}$. 

In [14]:
# --- 2. Bigram counts: X[j, i] = x_{j|i} = # times word j follows word i ---
X = np.zeros((k, k), dtype=np.int64)
for t in range(1, n):
    i_prev = w2i[tokens[t - 1]]  # i = previous word
    j_next = w2i[tokens[t]]      # j = next word
    X[j_next, i_prev] += 1

# x_i = sum_j x_{j|i}  (times i appears as the previous word)
x_prev = X.sum(axis=0)  # shape (k,)

print(X)
print(x_prev)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
[103  52   8   2   1   1   1   1   5   1   4   1   2   2  27   1  10   1
   1   1   1   2   2  65   3   1   1   1   4   2   1  14   2   1  11   1
   1   1   2   1   2   4  14   1   2   1   1   1   5   3   1   1   2   2
   1   1   2   1   1   4   4   2   1   1   1   1   1   1   1  22   1   6
   3   3   2   3   1   1   1   4   1   1   1   1   1   1   1   1   1   2
   3   1   1   2   1   1   1   2   8   1   1   1   1   1   1   2   1   2
   1   1   1   1   1   1   1   1   3   1   3   1   2   1   1   4   1   4
   1   1   1   1   1   9   1   7   2   1  22   1   1   1   1   3   1   1
   1   1   2   2   7   1   1   6   1   1   2   1   3   4   1   1   1   1
   2   1   1   4   1   2   2   1   2   1   2   4   2   2   1   1   1   2
   2   1   2   4   1   1   1   1   1   1   1  21   2   1   7   1   1   1
   3   1   1   1   4   1   2   7   3   1   1   3   2   6   3   3   4   1
   2

The first column refers to the number of times each word appears after '.'. More specifically, after '.', 'i' appears 2 times, 'the' appears 6 times and 'you' appears 4 times. And the sum of these numbers is the number of times '.' appears as the previous word. 

The next function calculates the log-Evidence for given values of $a_1, \dots, a_k$. Note that Evidence is the probability of obtaining the bigram sequence given values of $a_1, \dots, a_k$. 

In [15]:
# --- 3. Log marginal likelihood (evidence) up to constants in a ---
# Here we parameterize a = (a_1,...,a_k) and A = sum_j a_j.
# The marginal over all rows i is:
#   prod_i [ Γ(A)/Γ(x_i + A) * prod_j Γ(x_{j|i}+a_j)/Γ(a_j) ]
# (multinomial coefficients omitted since they do not depend on a)

def log_evidence(a):
    A = a.sum()
    val = 0.0
    for i in range(k):
        if x_prev[i] == 0:
            continue
        val += gammaln(A) - gammaln(x_prev[i] + A)
        val += (gammaln(X[:, i] + a) - gammaln(a)).sum()
    return val

The next function calculates the gradient of the log-evidence using the digamma function (which is in-built in scipy, and is the derivative of the logarithm of the Gamma function). The notes has the exact formula for the gradient in terms of the digamma function. 

In [16]:
def grad_log_evidence(a):
    A = a.sum()
    g = np.zeros(k)
    for i in range(k):
        if x_prev[i] == 0:
            continue
        # vector part: sum_i [ ψ(x_{j|i}+a_j) - ψ(a_j) ]
        g += digamma(X[:, i] + a) - digamma(a)
        # scalar part from A: sum_i [ ψ(A) - ψ(x_i + A) ] added to every component
        g += digamma(A) - digamma(x_prev[i] + A)
    return g


The code below maximizes the log-Evidence with respect to $a_1, \dots, a_k$. It uses a simple gradient ascent method. It calculates the gradient of the log-Evidence at the current value of $a = (a_1, \dots, a_k)$ and takes a step in that direction. A line search is done in that direction to find the best step-size. 

In [18]:
a = 1 * np.ones(k) #initial values of a_1, \dots, a_k

print("Optimizing hyperparameters a = (a_1,...,a_k)...")
for it in range(1000):
    log_a = np.log(a)
    le = log_evidence(a)
    g = grad_log_evidence(a) * a  # chain rule: d/d(log a) = a * d/da
    step = 0.01
    for _ in range(10):
        trial = np.exp(log_a + step * g).clip(1e-10)
        if log_evidence(trial) > le:
            break
        step *= 0.5
    a = np.exp(log_a + step * g).clip(1e-10)
    if it % 100 == 0 or it == 299:
        print(f"  iter {it:3d}: log_ev = {le:.2f}, A = {a.sum():.3f}")

A = a.sum()


Optimizing hyperparameters a = (a_1,...,a_k)...
  iter   0: log_ev = -14053.72, A = 608.212
  iter 100: log_ev = -12182.71, A = 905.970
  iter 200: log_ev = -12162.48, A = 774.779
  iter 299: log_ev = -12140.97, A = 639.746
  iter 300: log_ev = -12140.73, A = 638.437
  iter 400: log_ev = -12114.75, A = 514.897
  iter 500: log_ev = -12084.10, A = 407.493
  iter 600: log_ev = -12048.90, A = 317.274
  iter 700: log_ev = -12010.07, A = 244.202
  iter 800: log_ev = -11969.50, A = 187.205
  iter 900: log_ev = -11929.86, A = 144.299


Below are the values of the estimated $a_1, \dots, a_k$ (and their sum $A = a_1 + \dots + a_k$). 

In [19]:
print(f"\nOptimal A = {A:.3f}\n")
print(f"{'word':>8s}  {'a_j':>10s}")
print("-" * 30)
for j in range(k):
    print(f"{vocab[j]:>8s}  {a[j]:10.4f}")


Optimal A = 113.206

    word         a_j
------------------------------
       .      5.4442
       a      2.4610
   about      0.4471
  across      0.1212
actually      0.0671
  advice      0.0671
  advise      0.0671
  afraid      0.0671
   after      0.2841
afterwards      0.0671
   again      0.1844
 against      0.0671
     air      0.0714
    alas      0.0714
   alice      1.0523
  alices      0.0671
     all      0.5204
  almost      0.0671
   along      0.0671
   aloud      0.0671
altogether      0.0671
   among      0.1212
      an      0.1212
     and      3.1297
 another      0.1754
  answer      0.0671
antipathies      0.0671
anxiously      0.0671
     any      0.1844
anything      0.1212
     are      0.0671
      as      0.7424
     ask      0.0714
  asking      0.0671
      at      0.5402
     ate      0.0671
australia      0.0671
    away      0.0671
    back      0.0714
    bank      0.0671
     bat      0.0714
    bats      0.1318
      be      0.3641
  beasts      

Having obtained $a_1, \dots, a_k$, we compute below the posterior mean estimates of $p_{j \mid i}$. 

In [20]:
P_hat = np.zeros((k, k))
for i in range(k):
    P_hat[:, i] = (X[:, i] + a) / (x_prev[i] + A)
print(P_hat)

[[0.02518083 0.0329543  0.04491734 ... 0.04767046 0.04767046 0.04767046]
 [0.01138253 0.01489639 0.02030406 ... 0.02154856 0.02154856 0.02154856]
 [0.00206812 0.00270655 0.00368909 ... 0.0039152  0.0039152  0.0039152 ]
 ...
 [0.0003102  0.00040596 0.00055332 ... 0.00058724 0.00058724 0.00058724]
 [0.0003102  0.00040596 0.00055332 ... 0.00058724 0.00058724 0.00058724]
 [0.0003102  0.00040596 0.00055332 ... 0.00058724 0.00058724 0.00058724]]


In [22]:
print(f"\n{'='*45}")
print("NEXT-WORD PREDICTIONS (posterior mean)")
print(f"{'='*45}")

#The following code predicts the top five most probable next words for a given word
def predict_next(prev_word, top_k=5):
    i = w2i[prev_word]
    probs = P_hat[:, i]                 # probs over j given i
    ranked = np.argsort(probs)[::-1]
    return [(i2w[j], probs[j]) for j in ranked[:top_k]]


#for ctx in ['the', 'cat', 'you', 'dog', '.', 'on']:
#    i = w2i[ctx]
#    lam = A / (x_prev[i] + A)  # shrinkage weight on the prior mean m
#    print(f"\nAfter '{ctx}'  (x_i={int(x_prev[i])}, λ={lam:.3f}):")
#    for word, prob in predict_next(ctx, top_k=5):
#        bar = '█' * int(prob * 40)
#        print(f"  {word:>8s}  {prob:.3f}  {bar}")



NEXT-WORD PREDICTIONS (posterior mean)


The code below generates sentences using this model. It takes as input a starting word (e.g., '.') and then starts generating from there until the period '.' appears. Sentence generation can be done in two different ways here. The first way uses the Bayesian estimates of $p_{j \mid i}$ and generates words from multinomials. The second way also generates $p_{j \mid i}$ from their posterior first before generating from multinomials. 

In [24]:
def generate(start_word, length=25, seed=45):
    rng = np.random.default_rng(seed)
    words = [start_word]
    for _ in range(length):
        i = w2i[words[-1]]
        j = rng.choice(k, p=P_hat[:, i])
        words.append(i2w[j])
        if words[-1] == '.':
            break
    return ' '.join(words)

print(f"\n{'='*45}")
print("GENERATED SENTENCES")
print(f"{'='*45}\n")
for s in range(20):
    print("  " + generate('.', seed=45 + s))


GENERATED SENTENCES

  . oh middle that there no the their on is several poison thought so much which listen they up out a afterwards and stopping .
  . up alice filled please was all red the middle tears to hurry the air smaller thats a bat hurry garden didnt through .
  . so suddenly know alice white getting candle to in behind such the of the practice it for shutting up but over .
  . however or longitude she seemed to look me the door all alice life hot but her roof she the garden .
  . here on in pink respectable .
  . the to ive wise distance rat a mouse ill eat would i happens and begun work of to .
  . were filled dozing cut the door the so it me very that bank suddenly if you dark in was nothing so passed cut a as
  . put of of .
  . .
  . dinah moment about of among just sadly and well was lying sight asking beginning the i tired way creep schoolroom set those and .
  . there walk curiosity did looked taste she was again larger half opened .
  . shut .
  . she locked three h

You can ignore the starting period while interpreting each of these sentences. 

In the above, we fixed the Bayesian estimate of $P[i, j]$ while generating the sentences. We can also draw $P$ from the correct Bayesian posterior, before generating each sentence. This is done below.



In [27]:
def sample_P_posterior(rng):
    """
    Sample a full transition matrix P from the posterior.
    Each column i is p_{·|i} ~ Dirichlet(X[:, i] + a).
    """
    P = np.zeros((k, k))
    for i in range(k):
        alpha = X[:, i] + a
        P[:, i] = rng.dirichlet(alpha)
    return P

def generate_from_P(start_word, P, length=25, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    words = [start_word]
    for _ in range(length):
        i = w2i[words[-1]]
        j = rng.choice(k, p=P[:, i])
        words.append(i2w[j])
        if words[-1] == '.':
            break
    return ' '.join(words)

def generate_posterior_sampled(start_word, length=25, seed=45):
    """
    For THIS sentence: sample P ~ posterior, then generate using that P.
    """
    rng = np.random.default_rng(seed)
    P = sample_P_posterior(rng)
    return generate_from_P(start_word, P, length=length, rng=rng)

# --- 9. Generate sentences ---
print(f"\n{'='*45}")
print("GENERATED SENTENCES (posterior-sampled P per sentence)")
print(f"{'='*45}\n")
for s in range(20):
    print(" " + generate_posterior_sampled('.', seed=45 + s))


GENERATED SENTENCES (posterior-sampled P per sentence)

 . she later out to which how few paper alice before .
 . size walk with followed in through of out inches new and you to leave it except you to her get her well and .
 . first pineapple and wander going they they but see saying think be it into poker it .
 . such quite again for eat it will cats in creep dozing a straight this the cut of her dear away it here shelves have .
 . first in bring bottle advise three thats down for drink dont what her of having up at this you time for getting it be with
 . a thing eyes bright but nice round considering a and there was not little a turkey out going pictures passage hanging a very pretending words
 . dinah you so trouble worth the garden still hot tired showing by do up such but cats the rabbit know from so .
 . to say all and to do hanging her sitting at wind into would use in to shrink sight or when australia hope twice think .
 . there long of very downward one now into alices wise a

Overall these sentences are more realistic compared to the simple i.i.d (bag of words) model. Obviously, more sophisticated language models will  yield much more realistic sentences. 